# Getting Started with Claude Fable 5 on Amazon Bedrock

**Anthropic's next-generation model for complex knowledge work, coding, and sustained autonomous operation across multi-day tasks.**

Claude Fable 5 plans across stages, delegates to sub-agents, and self-verifies its work. It is designed for long-horizon agentic workflows that require deep reasoning and autonomous execution.

---

## What you'll learn

- **Access setup**: Opt in to `provider_data_share` data retention (required before first use)
- **Basic invocation**: Call Claude Fable 5 using the Messages API, InvokeModel, and Converse APIs
- **Adaptive thinking**: Leverage Claude's intelligent reasoning with configurable effort levels
- **Handling refusals**: Gracefully handle content classifier blocks (`stop_reason: "refusal"`)
- **Fallback credit (beta)**: Avoid double-paying cache costs when retrying refused requests on another model
- **Prompt caching**: Reduce costs on repeated prefixes

---

## Key capabilities

| Capability | Details |
|------------|--------|
| **Model ID** | `anthropic.claude-fable-5` |
| **Context window** | 1,024,000 tokens (1M) |
| **Max output** | 128,000 tokens |
| **Reasoning** | Adaptive thinking only (`thinking.type: "adaptive"`) |
| **Input modalities** | Text, Image |
| **Output modalities** | Text |
| **Prompt caching** | Yes — min 1,024 tokens per checkpoint, max 4 checkpoints, TTL 5 min or 1 hour |
| **Knowledge cutoff** | January 2026 |
| **Launch date** | June 9, 2026 |

---

## When to use Claude Fable 5

- **Multi-day autonomous coding** — architecture, implementation, testing, and deployment in one session
- **Complex knowledge work** — synthesizing long documents, research, and structured deliverables
- **Agentic workflows** — tool use with planning, delegation, and self-verification
- **Tasks requiring deep reasoning** — math, logic, analysis, and multi-step problem solving

---

## Important: Access prerequisites

Claude Fable 5 is an **opt-in-only model**. Before you can invoke it, you must:

1. Have an AWS account
2. **Opt in to `provider_data_share`** data retention mode (shown in Section 2 below)

Without the opt-in, the model returns `status: "unavailable"`.

---

## Available regions

| Region | 
|--------|
| us-east-1 (N. Virginia) | 
| eu-north-1 (Stockholm) |  

For the latest list of supported regions, see the [Claude Fable 5 model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-fable-5.html).


> **Note:** In-region calls use the bare model ID (`anthropic.claude-fable-5`). Cross-region calls require the geo-prefixed CRIS ID (e.g., `us.anthropic.claude-fable-5`). The Messages API (Bedrock Mantle) always uses the bare model ID.

---

## 1. Setup and configuration

In [ ]:
# Install required packages (uncomment if needed)
# !pip install boto3 --upgrade
# !pip install "anthropic[bedrock]" --upgrade

In [ ]:
import boto3
import json
import requests
from botocore.config import Config
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

# Try importing Anthropic Bedrock Mantle client
try:
    from anthropic import AnthropicBedrockMantle
    MANTLE_AVAILABLE = True
except ImportError:
    print(
        "Note: anthropic[bedrock] not installed. "
        "Messages API examples will be skipped. "
        "Install with: pip install \"anthropic[bedrock]\""
    )
    MANTLE_AVAILABLE = False

# Configuration
REGION = "us-east-1"

# Model IDs
MODEL_ID = "anthropic.claude-fable-5"
MANTLE_BASE_URL = f"https://bedrock-mantle.{REGION}.api.aws"

# Cross-Region Inference (CRIS) model ID — required for InvokeModel/Converse
# In-region (us-east-1, eu-north-1): bare model ID works - required for Messages API
_REGION_TO_CRIS_PREFIX = {
    "us-east-1": "us",
    "us-east-2": "us",
    "us-west-2": "us",
    "eu-north-1": "eu",
    "eu-west-1": "eu",
    "ap-northeast-1": "global", "ap-southeast-4": "global",
}
CRIS_PREFIX = _REGION_TO_CRIS_PREFIX.get(REGION)
CRIS_MODEL_ID = f"{CRIS_PREFIX}.{MODEL_ID}" if CRIS_PREFIX else MODEL_ID

# Longer timeout for thinking responses
FAST_CONFIG = Config(read_timeout=1200, retries={"max_attempts": 1})

# Initialize clients
session = boto3.Session()
bedrock_runtime = session.client(
    service_name="bedrock-runtime",
    region_name=REGION,
    config=FAST_CONFIG,
)

if MANTLE_AVAILABLE:
    mantle_client = AnthropicBedrockMantle(
        aws_region=REGION,
        base_url=f"{MANTLE_BASE_URL}/anthropic",
    )

print(f"✓ Region:          {REGION}")
print(f"✓ Model ID:        {MODEL_ID}")
print(f"✓ CRIS Model ID:   {CRIS_MODEL_ID} (for InvokeModel/Converse)")
print(f"✓ Mantle Base URL: {MANTLE_BASE_URL}")
print(f"✓ Mantle SDK:      {'available' if MANTLE_AVAILABLE else 'not installed'}")
print(f"✓ Client initialized successfully")

---

## 2. Data retention opt-in (REQUIRED)

Claude Fable 5's only allowed data retention mode is `provider_data_share`. This signals consent to share request data with Anthropic for abuse detection. Until you explicitly set this mode, the model returns `status: "unavailable"`.

**What `provider_data_share` means:** Prompts and outputs are retained for up to 30 days. As required by Anthropic, you must opt in to sharing retained traffic with Anthropic for their abuse detection processing and potential human review.

> ⚠️ **Important: You can set the data retention opt-in in one or both endpoints depending on the APIs you'll use.**:
> - **Bedrock Mantle** (`PUT https://bedrock-mantle.{region}.api.aws/v1/data_retention`) — read by the Messages API
> - **Bedrock control plane** (`PUT https://bedrock.{region}.amazonaws.com/data-retention`) — read by InvokeModel / Converse
>
> If you only set the Mantle store, Messages API works but InvokeModel/Converse will reject the request.


In [ ]:
# Step 1: Opt in to provider_data_share (account scope)
# This must be done ONCE before you can invoke Claude Fable 5.
# You can set the data retention opt-in in one or both endpoints depending on the APIs you'll use.

def opt_in_data_retention(region: str = REGION):
    """Set data retention mode to provider_data_share on BOTH control planes.

    The Mantle store (bedrock-mantle.{region}.api.aws) is read by the Messages API.
    The control-plane store (bedrock.{region}.amazonaws.com) is read by InvokeModel/Converse.
    Both must be set for all API paths to work.
    """
    credentials = session.get_credentials().get_frozen_credentials()
    body = json.dumps({"mode": "provider_data_share"})

    # Store 1: Bedrock Mantle (for Messages API)
    mantle_url = f"https://bedrock-mantle.{region}.api.aws/v1/data_retention"
    req = AWSRequest(
        method="PUT",
        url=mantle_url,
        data=body,
        headers={"Content-Type": "application/json"}
    )
    SigV4Auth(credentials, "bedrock-mantle", region).add_auth(req)
    resp1 = requests.put(mantle_url, headers=dict(req.headers), data=body)
    print(f"Mantle store:        {resp1.status_code} - {resp1.text}")

    # Store 2: Bedrock control plane (for InvokeModel / Converse)
    cp_url = f"https://bedrock.{region}.amazonaws.com/data-retention"
    req2 = AWSRequest(
        method="PUT",
        url=cp_url,
        data=body,
        headers={"Content-Type": "application/json"}
    )
    SigV4Auth(credentials, "bedrock", region).add_auth(req2)
    resp2 = requests.put(cp_url, headers=dict(req2.headers), data=body)
    print(f"Control-plane store: {resp2.status_code} - {resp2.text}")

    return resp1, resp2

# Uncomment to execute the opt-in:
opt_in_data_retention()

In [ ]:
# Step 2: Verify model availability after opt-in

def check_model_availability(region: str = REGION):
    """Check if Claude Fable 5 is available for your account."""
    credentials = session.get_credentials().get_frozen_credentials()
    url = f"https://bedrock-mantle.{region}.api.aws/v1/models/{MODEL_ID}"

    req = AWSRequest(
        method="GET",
        url=url,
        headers={"Content-Type": "application/json"}
    )
    SigV4Auth(credentials, "bedrock-mantle", region).add_auth(req)

    resp = requests.get(url, headers=dict(req.headers))
    result = resp.json()
    print(json.dumps(result, indent=2))

    # Expected when opted in:
    # {
    #   "id": "anthropic.claude-fable-5",
    #   "status": "available",
    #   "data_retention": {
    #     "mode": "provider_data_share",
    #     "source": "account",
    #     "allowed_modes": ["provider_data_share"]
    #   }
    # }
    return result

# Uncomment to check availability:
check_model_availability()

### Project-scope opt-in (alternative)

You can also opt in at the project level instead of account level:

In [ ]:
# Project-scope opt-in (alternative to account-scope)

def opt_in_project_scope(project_id: str, region: str = REGION):
    """Set data retention mode to provider_data_share for a specific project."""
    credentials = session.get_credentials().get_frozen_credentials()
    url = f"https://bedrock-mantle.{region}.api.aws/v1/organization/projects/{project_id}"
    body = json.dumps({"data_retention": {"mode": "provider_data_share"}})

    req = AWSRequest(
        method="POST",
        url=url,
        data=body,
        headers={"Content-Type": "application/json"}
    )
    SigV4Auth(credentials, "bedrock-mantle", region).add_auth(req)

    resp = requests.post(url, headers=dict(req.headers), data=body)
    print(f"Status: {resp.status_code}")
    print(f"Response: {resp.text}")
    return resp

# Uncomment and provide your project ID:
# opt_in_project_scope("your-project-id-here")

---

## 3. Basic usage with Messages API (Bedrock Mantle)

The Messages API via Bedrock Mantle is the **recommended** way to use Claude Fable 5. It uses Anthropic's native request/response format, authenticated with AWS SigV4.

> ⚠️ **Important:** Claude Fable 5 uses adaptive thinking by default, even when you don't pass the `thinking` parameter. This means responses always contain a `ThinkingBlock` before the `TextBlock`. Always iterate over `content` blocks and filter by `type == "text"` to extract the response text.

In [ ]:
# Messages API - basic request via AnthropicBedrockMantle SDK
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MODEL_ID,
            max_tokens=2048,
            messages=[
                {
                    "role": "user",
                    "content": "What are three key considerations when designing a fault-tolerant distributed system?"
                }
            ],
        )

        # Filter for text blocks (response may include ThinkingBlocks which have no .text)
        for block in message.content:
            if block.type == "text":
                print(block.text)

        print(f"\n📊 Usage: Input={message.usage.input_tokens}, Output={message.usage.output_tokens}")
        print(f"Stop reason: {message.stop_reason}")

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock] for Messages API support")

---

## 4. Basic usage with InvokeModel API

The InvokeModel API provides direct access to Claude's native request format via the `bedrock-runtime` endpoint.

> ⚠️ **CRIS model ID required for cross-region calls.** InvokeModel/Converse require the geo-prefixed CRIS model ID (e.g., `us.anthropic.claude-fable-5`).

In [ ]:
# InvokeModel API - basic request
# Note: uses CRIS_MODEL_ID (geo-prefixed) for bedrock-runtime
request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 4096,
    "messages": [
        {
            "role": "user",
            "content": "Explain the CAP theorem and its practical implications for microservices architectures."
        }
    ]
}

try:
    response = bedrock_runtime.invoke_model(
        modelId=CRIS_MODEL_ID,
        contentType="application/json",
        accept="application/json",
        body=json.dumps(request_body)
    )

    result = json.loads(response["body"].read())

    # Filter for text blocks (response may include thinking blocks)
    for block in result["content"]:
        if block["type"] == "text":
            print(block["text"])

    print(f"\n📊 Usage: Input={result['usage']['input_tokens']}, Output={result['usage']['output_tokens']}")
    print(f"Stop reason: {result['stop_reason']}")

except Exception as e:
    print(f"Error: {e}")

---

## 5. Basic usage with Converse API

The Converse API provides a unified interface across Amazon Bedrock models.

In [ ]:
# Converse API - basic request (uses CRIS_MODEL_ID)
try:
    response = bedrock_runtime.converse(
        modelId=CRIS_MODEL_ID,
        messages=[
            {
                "role": "user",
                "content": [{"text": "What are the SOLID principles in software engineering? Be concise."}]
            }
        ],
        inferenceConfig={
            "maxTokens": 2048,
            "temperature": 1  # Claude Fable 5: temperature must be 1.0 or unset
        }
    )

    # Iterate over content blocks — may include thinking blocks
    for block in response["output"]["message"]["content"]:
        if block.get("text"):
            print(block["text"])
    print(f"\n📊 Usage: Input={response['usage']['inputTokens']}, Output={response['usage']['outputTokens']}")

except Exception as e:
    print(f"Error: {e}")

---

## 6. Adaptive thinking

Claude Fable 5 uses **adaptive thinking exclusively**. It does NOT support manual extended thinking (`thinking.type: "enabled"` with `budget_tokens`) or disabled thinking (`thinking.type: "disabled"`). Both will return a 400 error.

With adaptive thinking, Claude decides whether and how deeply to reason based on task complexity. You can guide this with the `effort` parameter.

### Effort levels

| Effort | Behavior | Best for |
|--------|----------|----------|
| **high** | Deep reasoning (default) | Complex analytical tasks |
| **medium** | Balanced — may skip thinking for simple queries | Mixed workloads |
| **low** | Minimal thinking, prioritizes speed | Simple queries, cost-sensitive |

### Sampling constraints

| Parameter | Requirement |
|-----------|-------------|
| `temperature` | Must be 1.0 or unset |
| `top_p` | Must be ≥ 0.99 and < 1.0, or unset |
| `top_k` | Not supported |

> ⚠️ **Migration from extended thinking:** Replace `{"type": "enabled", "budget_tokens": N}` with `{"type": "adaptive"}` and use `output_config.effort` to control thinking depth. Remove `{"type": "disabled"}` — omitting the thinking parameter entirely gives you adaptive thinking by default.

In [ ]:
# Adaptive thinking with Messages API
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MODEL_ID,
            max_tokens=16000,
            thinking={"type": "adaptive"},
            extra_body={"output_config": {"effort": "high"}},
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Design a lock-free concurrent hash map that supports "
                        "both read-heavy and write-heavy workloads. Explain the "
                        "memory reclamation strategy and prove it is ABA-safe."
                    ),
                }
            ],
        )

        # Check if thinking was triggered
        has_thinking = any(block.type == "thinking" for block in message.content)
        print(f"Claude decided to think: {has_thinking}\n")

        for block in message.content:
            if block.type == "thinking":
                sig_len = len(block.signature) if block.signature else 0
                print(f"🧠 [Thinking block] Signature: {sig_len} chars")
                print("   (Summarized thinking — signature proves reasoning occurred)\n")
            elif block.type == "text":
                text = block.text
                print(f"💬 Response:\n{text[:1000]}..." if len(text) > 1000 else f"💬 Response:\n{text}")

        print(f"\n📊 Usage: Input={message.usage.input_tokens}, Output={message.usage.output_tokens}")

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

In [ ]:
# Adaptive thinking with InvokeModel API
request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 16000,
    "thinking": {
        "type": "adaptive"
    },
    "output_config": {
        "effort": "high"
    },
    "messages": [
        {
            "role": "user",
            "content": "Solve this step by step: what is 27^(1/3) * 16^(1/4)?"
        }
    ]
}

try:
    response = bedrock_runtime.invoke_model(
        modelId=CRIS_MODEL_ID,
        contentType="application/json",
        accept="application/json",
        body=json.dumps(request_body)
    )

    result = json.loads(response["body"].read())

    has_thinking = any(block["type"] == "thinking" for block in result["content"])
    print(f"Claude decided to think: {has_thinking}\n")

    for block in result["content"]:
        if block["type"] == "thinking":
            print("🧠 [Thinking block detected]\n")
        elif block["type"] == "text":
            print(f"💬 Response:\n{block['text']}")

    print(f"\n📊 Usage: Input={result['usage']['input_tokens']}, Output={result['usage']['output_tokens']}")

except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Adaptive thinking with Converse API
try:
    response = bedrock_runtime.converse(
        modelId=CRIS_MODEL_ID,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "text": (
                            "Analyze the trade-offs between event sourcing and CRUD "
                            "for a high-volume financial transaction system. "
                            "Consider consistency, auditability, and operational complexity."
                        )
                    }
                ]
            }
        ],
        inferenceConfig={
            "maxTokens": 16000,
            "temperature": 1
        },
        additionalModelRequestFields={
            "thinking": {
                "type": "adaptive"
            },
            "output_config": {
                "effort": "medium"
            }
        }
    )

    output_message = response["output"]["message"]
    has_thinking = any(block.get("type") == "thinking" for block in output_message["content"])
    print(f"Effort level: medium")
    print(f"Claude decided to think: {has_thinking}\n")

    for block in output_message["content"]:
        if block.get("type") == "thinking":
            print("🧠 [Thinking block detected]\n")
        elif block.get("text"):
            text = block["text"]
            print(f"💬 Response:\n{text[:800]}..." if len(text) > 800 else f"💬 Response:\n{text}")

    print(f"\n📊 Usage: Input={response['usage']['inputTokens']}, Output={response['usage']['outputTokens']}")

except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Comparing effort levels on the same prompt
def test_effort_level(effort_level: str, prompt: str):
    """Test a specific effort level and return stats."""
    if not MANTLE_AVAILABLE:
        return {"effort": effort_level, "error": "SDK not installed"}

    try:
        message = mantle_client.messages.create(
            model=MODEL_ID,
            max_tokens=12000,
            thinking={"type": "adaptive"},
            extra_body={"output_config": {"effort": effort_level}},
            messages=[{"role": "user", "content": prompt}],
        )

        has_thinking = any(b.type == "thinking" for b in message.content)
        response_len = sum(len(b.text) for b in message.content if b.type == "text")

        return {
            "effort": effort_level,
            "thought": has_thinking,
            "input_tokens": message.usage.input_tokens,
            "output_tokens": message.usage.output_tokens,
            "response_chars": response_len,
        }
    except Exception as e:
        return {"effort": effort_level, "error": str(e)}


prompt = (
    "Prove that for any continuous function f on [0,1] with f(0)=0 and f(1)=1, "
    "there exists a point c in (0,1) such that f(c) = c. "
    "Then explain why this does not generalize to open intervals."
)

print("Testing different effort levels on the same prompt...\n")
print(f"{'Effort':<8} {'Thought':<8} {'Input':<8} {'Output':<8} {'Response':<10}")
print("-" * 50)

for effort in ["low", "medium", "high"]:
    result = test_effort_level(effort, prompt)
    if "error" not in result:
        print(
            f"{result['effort']:<8} "
            f"{str(result['thought']):<8} "
            f"{result['input_tokens']:<8} "
            f"{result['output_tokens']:<8} "
            f"{result['response_chars']:<10}"
        )
    else:
        print(f"{result['effort']:<8} Error: {result['error']}")

print("\n✓ Higher effort = deeper reasoning, more tokens, better quality on complex tasks")

---

## 7. Handling refusals

Claude Fable 5 includes **blocking classifiers for dual-use content** in cybersecurity and biology. Refusal rates are **materially higher** than on previous Claude models.

When a classifier blocks a request:
- The API returns HTTP 200 with `stop_reason: "refusal"`
- A `stop_details` object contains the restriction category and explanation
- **Prompt-stage refusals** (blocked before inference begins) are **not billed**
- **Mid-stream refusals** (blocked after partial output) are billed for tokens generated before the block

### Response shape on refusal

```json
{
  "stop_reason": "refusal",
  "stop_details": {
    "type": "refusal",
    "category": "cyber",  // or "bio", or null
    "explanation": "This request triggered restrictions..."
  }
}
```

> **Best practice:** Always branch on `stop_reason`, not on `stop_details`. The `stop_details` field is informational and can be null even when `stop_reason` is `"refusal"`.

In [ ]:
# Handling refusals gracefully

def invoke_with_refusal_handling(messages: list, max_tokens: int = 4096):
    """
    Invoke Claude Fable 5 with proper refusal handling.
    Returns (response_text, was_refused, stop_details).
    """
    if not MANTLE_AVAILABLE:
        print("SDK not available")
        return None, False, None

    try:
        message = mantle_client.messages.create(
            model=MODEL_ID,
            max_tokens=max_tokens,
            messages=messages,
        )

        if message.stop_reason == "refusal":
            # Handle refusal
            stop_details = getattr(message, "stop_details", None)
            category = getattr(stop_details, "category", "unknown") if stop_details else "unknown"
            explanation = getattr(stop_details, "explanation", "No explanation provided") if stop_details else "No explanation provided"

            print(f"⚠️  Request REFUSED")
            print(f"   Category: {category}")
            print(f"   Explanation: {explanation}")
            print(f"   Input tokens billed: {message.usage.input_tokens}")
            print(f"   Output tokens billed: {message.usage.output_tokens}")

            # Check for partial content (mid-stream refusal)
            partial_text = ""
            for block in message.content:
                if hasattr(block, "text") and block.text:
                    partial_text += block.text

            if partial_text:
                print(f"   Partial content received: {len(partial_text)} chars")

            return partial_text, True, stop_details

        else:
            # Normal response
            response_text = ""
            for block in message.content:
                if hasattr(block, "text"):
                    response_text += block.text

            print(f"✓ Response received ({message.stop_reason})")
            print(f"  Tokens: Input={message.usage.input_tokens}, Output={message.usage.output_tokens}")
            return response_text, False, None

    except Exception as e:
        print(f"Error: {e}")
        return None, False, None


# Example: A normal request that should succeed
print("=" * 50)
print("Test 1: Normal request")
print("=" * 50)
text, refused, details = invoke_with_refusal_handling(
    [{"role": "user", "content": "Explain how TLS 1.3 handshake works."}]
)
if not refused and text:
    print(f"\n{text[:500]}...")

# Example: B request that should fail
print("=" * 50)
print("Test 1: Normal request")
print("=" * 50)
text, refused, details = invoke_with_refusal_handling(
    [{"role": "user", "content": "Your prompt here"}]
)
if not refused and text:
    print(f"\n{text[:500]}...")

In [ ]:
# Pattern: Retry on a fallback model after refusal

def invoke_with_fallback(messages: list, system: str = None, tools: list = None,
                         primary_model: str = MODEL_ID, fallback_model: str = "anthropic.claude-opus-4-8"):
    """
    Try primary model (Fable 5), fall back to another model on refusal.
    """
    if not MANTLE_AVAILABLE:
        print("SDK not available")
        return None

    # Build the request body
    request_kwargs = {
        "model": MODEL_ID,
        "max_tokens": 4096,
        "messages": messages,
        "extra_body": {
            "anthropic_beta": ["fallback-credit-2026-06-01"],
        },
    }

    if system:
        request_kwargs["system"] = system
    if tools:
        request_kwargs["tools"] = tools

    try:
        # Try primary model
        message = mantle_client.messages.create(
            model=primary_model,
            max_tokens=4096,
            messages=request_kwargs,
        )

        if message.stop_reason == "refusal":
            print(f"⚠️  {primary_model} refused the request.")
            print(f"   Falling back to {fallback_model}...\n")
            details = getattr(message, "stop_details", None)
            token = getattr(details, "fallback_credit_token", None) if details else None
            category = getattr(details, "category", "unknown") if details else "unknown"
            print(message)
            print(details)
            print(token)
            print(category)

        else:
            print(f"✓ Primary model responded successfully")
            return next((b.text for b in message.content if b.type == "text"), None)

    except Exception as e:
        print(f"Error: {e}")
        return None

# Example usage:
result = invoke_with_fallback(
     [{"role": "user", "content": "How to do gene mutation with cancer"}]
 )

---

## 8. Fallback credit for refused requests (beta)

When Claude Fable 5 refuses a request, you may want to retry on another model (e.g., Claude Opus 4.8). Normally you'd re-pay cache-write costs for the conversation prefix. **Fallback credit** eliminates this double-charge by issuing a one-time credit token on refusal.

### Requirements
- Beta flag: `fallback-credit-2026-06-01` in `anthropic_beta` array on **both** the original request and retry
- Token must be redeemed within **5 minutes**
- Request body (`system`, `messages`, `tools`) must match exactly on retry
- Target model must be a valid fallback target of the source model

### How it works
1. Send request to Fable 5 with the beta flag → receive refusal with `fallback_credit_token`
2. Retry to fallback model with same body + the token → get credit for overlapping cached prefix

In [ ]:
# Fallback credit workflow (beta)
# Uses the Messages API via Bedrock Mantle (AnthropicBedrockMantle client)

from anthropic import BadRequestError

FALLBACK_MODEL = "anthropic.claude-opus-4-8"
FALLBACK_BETA = "fallback-credit-2026-06-01"


def _send(model: str, body: dict):
    """Send a request via the Bedrock Mantle beta endpoint."""
    return mantle_client.beta.messages.create(
        model=model,
        betas=[FALLBACK_BETA],
        **body,
    )


def invoke_with_fallback_credit(messages: list, system: str = None,
                                tools: list = None, max_tokens: int = 4096):
    """
    Invoke Claude Fable 5 with fallback credit support.
    On refusal, retry on fallback model using the 3-tier strategy:
      1. Prefill continuation + credit token
      2. Credit token without prefill
      3. Plain retry without token
    """
    if not MANTLE_AVAILABLE:
        print("SDK not available")
        return None

    # Build the base request body
    request = {"max_tokens": max_tokens, "messages": messages}
    if system:
        request["system"] = system
    if tools:
        request["tools"] = tools

    # Step 1: Send to Claude Fable 5
    response = _send(MODEL_ID, request)

    if response.stop_reason != "refusal":
        print(f"✓ Request succeeded (no refusal)")
        return next((b.text for b in response.content if b.type == "text"), None)

    # Step 2: Extract fallback credit token
    details = getattr(response, "stop_details", None)
    token = getattr(details, "fallback_credit_token", None) if details else None
    category = getattr(details, "category", "unknown") if details else "unknown"

    print(f"⚠️  Refusal received.")
    print(f"   Category: {category}")
    print(f"   Fallback credit token: {'received' if token else 'null (not available)'}")

    if not token:
        # No token — plain retry on fallback model
        print(f"   Retrying on {FALLBACK_MODEL} without credit token...")
        response = _send(FALLBACK_MODEL, request)
        return next((b.text for b in response.content if b.type == "text"), None)

    # Build the exact body with token (token goes at body level)
    exact_body = {**request, "fallback_credit_token": token}

    # Step 3: Try prefill continuation if the claim allows it
    has_prefill_claim = getattr(details, "fallback_has_prefill_claim", None)

    if has_prefill_claim is not False:
        # Echo the refusal's content blocks as assistant prefill.
        # Strip trailing whitespace from the final text block
        # (prefill validator rejects it; the server-side match tolerates the edit).
        echoed = [block.model_dump() for block in response.content]
        if echoed and echoed[-1].get("type") == "text":
            echoed[-1]["text"] = echoed[-1]["text"].rstrip()

        attempt = {
            **exact_body,
            "messages": [
                *request["messages"],
                {"role": "assistant", "content": echoed},
            ],
        }

        try:
            print(f"   Retrying on {FALLBACK_MODEL} with prefill continuation + token...")
            response = _send(FALLBACK_MODEL, attempt)
            print(f"✓ Fallback succeeded (prefill continuation)")
            return next((b.text for b in response.content if b.type == "text"), None)
        except BadRequestError as e:
            if "redemption temporarily unavailable" in str(e):
                raise  # Transient — retry with the same token within 5 min window
            print(f"   Prefill continuation rejected, trying without prefill...")

    # Step 4: Retry with token but no prefill
    try:
        response = _send(FALLBACK_MODEL, exact_body)
        print(f"✓ Fallback succeeded (with credit token)")
        return next((b.text for b in response.content if b.type == "text"), None)
    except BadRequestError as e:
        if "redemption temporarily unavailable" in str(e):
            raise  # Transient — retry with the same token within 5 min window
        # Token rejected permanently — forfeit it and retry without
        print(f"   Credit token rejected. Retrying without token...")
        response = _send(FALLBACK_MODEL, request)
        return next((b.text for b in response.content if b.type == "text"), None)


# Example:
# result = invoke_with_fallback_credit(
#     messages=[{"role": "user", "content": "Your prompt here"}],
#     system="You are a helpful assistant."
# )

---

## 9. Prompt caching

Claude Fable 5 supports prompt caching to reduce costs on repeated conversation prefixes.

| Property | Value |
|----------|-------|
| Min tokens per cache checkpoint | 1,024 |
| Max cache checkpoints per request | 4 |
| Supported TTL | 5 minutes, 1 hour |
| Fields that accept cache checkpoints | `system`, `messages`, `tools` |

In [ ]:
# Prompt caching example with InvokeModel API

# A large system prompt that benefits from caching
system_prompt = """You are an expert software architect specializing in distributed systems, 
microservices, and cloud-native applications. You have deep expertise in:
- Event-driven architectures
- CQRS and event sourcing patterns
- Kubernetes orchestration
- Service mesh implementations
- Observability and distributed tracing
- Chaos engineering principles

When answering questions, always consider:
1. Scalability implications
2. Failure modes and recovery strategies
3. Operational complexity
4. Cost optimization
5. Security boundaries
"""  # In production, this would be much longer (1024+ tokens to be cache-eligible)

# First request — establishes the cache
request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 4096,
    "system": [
        {
            "type": "text",
            "text": system_prompt,
            "cache_control": {"type": "ephemeral"}  # Cache this block
        }
    ],
    "messages": [
        {
            "role": "user",
            "content": "How should I design a saga pattern for a payment processing workflow?"
        }
    ]
}

try:
    response = bedrock_runtime.invoke_model(
        modelId=CRIS_MODEL_ID,
        contentType="application/json",
        accept="application/json",
        body=json.dumps(request_body)
    )

    result = json.loads(response["body"].read())
    print("First request (cache write):")
    response_text = next((b['text'] for b in result['content'] if b['type'] == 'text'), '')
    print(f"  Response: {response_text[:300]}...")
    print(f"  Usage: {result['usage']}")
    # Look for cache_creation_input_tokens and cache_read_input_tokens in usage

except Exception as e:
    print(f"Error: {e}")

# Second request — same system prompt, reads from cache
request_body["messages"] = [
    {
        "role": "user",
        "content": "Now explain the compensating transaction pattern for rollbacks."
    }
]

try:
    response = bedrock_runtime.invoke_model(
        modelId=CRIS_MODEL_ID,
        contentType="application/json",
        accept="application/json",
        body=json.dumps(request_body)
    )

    result = json.loads(response["body"].read())
    print("\nSecond request (cache read):")
    response_text = next((b['text'] for b in result['content'] if b['type'] == 'text'), '')
    print(f"  Response: {response_text[:300]}...")
    print(f"  Usage: {result['usage']}")
    # cache_read_input_tokens should be populated

except Exception as e:
    print(f"Error: {e}")

---

## 10. Tool use with connector text summarization

Claude Fable 5 has a unique behavior: **text emitted between tool calls** ("connector text" like "Let me check that file next...") is **summarized server-side** and returned as a thinking block rather than a plain text content block.

### Customer impact
- Tool-use responses may contain additional **thinking blocks** where previous models emitted plain text between `tool_use` blocks
- Pass these thinking blocks back **unchanged** in multi-turn conversations (signature validated on passback)
- Final assistant answers (after all tool use is complete) remain plain text
- This feature is enabled server-side — no opt-in or opt-out

In [ ]:
# Tool use example with connector text summarization awareness

tools = [
    {
        "name": "search_docs",
        "description": "Search internal documentation for relevant information",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query"},
                "max_results": {"type": "integer", "description": "Max results to return", "default": 5}
            },
            "required": ["query"]
        }
    },
    {
        "name": "read_file",
        "description": "Read a file from the repository",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "File path"}
            },
            "required": ["path"]
        }
    },
    {
        "name": "write_file",
        "description": "Write content to a file",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "File path"},
                "content": {"type": "string", "description": "File content"}
            },
            "required": ["path", "content"]
        }
    }
]

if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MODEL_ID,
            max_tokens=4096,
            thinking={"type": "adaptive"},
            tools=tools,
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Search the docs for our authentication module, "
                        "then read the main auth file and suggest improvements "
                        "for JWT token rotation."
                    )
                }
            ],
        )

        print(f"Stop reason: {message.stop_reason}")
        print(f"Content blocks: {len(message.content)}\n")

        for i, block in enumerate(message.content):
            if block.type == "thinking":
                # This may be connector text that was summarized into a thinking block
                print(f"Block {i}: 🧠 Thinking (may be summarized connector text)")
                print(f"   Has signature: {bool(getattr(block, 'signature', None))}")
                print(f"   → Pass this block back unchanged in multi-turn conversations\n")
            elif block.type == "text":
                print(f"Block {i}: 💬 Text")
                print(f"   {block.text[:200]}...\n" if len(block.text) > 200 else f"   {block.text}\n")
            elif block.type == "tool_use":
                print(f"Block {i}: 🔧 Tool call: {block.name}")
                print(f"   Input: {json.dumps(block.input)[:150]}\n")

        print("\n📌 Note: In production, execute the tool calls, pass results back")
        print("   (including any thinking blocks) to continue the agentic loop.")

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

---

## 11. Streaming responses

Claude Fable 5 supports response streaming on both `bedrock-runtime` and `bedrock-mantle` endpoints. This is important for refusal handling — in streaming mode, a refusal arrives as the final `message_delta` event with `stop_reason: "refusal"`. Any content streamed before the refusal is valid partial output.

In [ ]:
# Streaming with Messages API
if MANTLE_AVAILABLE:
    try:
        print("Streaming response:\n")
        with mantle_client.messages.stream(
            model=MODEL_ID,
            max_tokens=2048,
            messages=[
                {
                    "role": "user",
                    "content": "Write a Python function that implements exponential backoff with jitter for API retries."
                }
            ],
        ) as stream:
            for text in stream.text_stream:
                print(text, end="", flush=True)

        # Get the final message for usage stats
        final_message = stream.get_final_message()
        print(f"\n\n📊 Usage: Input={final_message.usage.input_tokens}, Output={final_message.usage.output_tokens}")
        print(f"Stop reason: {final_message.stop_reason}")

        # Check for refusal in streaming
        if final_message.stop_reason == "refusal":
            print("\n⚠️  Stream ended with refusal. Partial content above is valid.")

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

In [ ]:
# Streaming with InvokeModel (invoke_model_with_response_stream)
try:
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 2048,
        "messages": [
            {
                "role": "user",
                "content": "Explain the difference between optimistic and pessimistic locking in databases."
            }
        ]
    }

    response = bedrock_runtime.invoke_model_with_response_stream(
        modelId=CRIS_MODEL_ID,
        contentType="application/json",
        accept="application/json",
        body=json.dumps(request_body)
    )

    print("Streaming response:\n")
    for event in response["body"]:
        chunk = json.loads(event["chunk"]["bytes"])
        if chunk["type"] == "content_block_delta":
            if chunk["delta"]["type"] == "text_delta":
                print(chunk["delta"]["text"], end="", flush=True)
        elif chunk["type"] == "message_delta":
            print(f"\n\nStop reason: {chunk['delta'].get('stop_reason', 'N/A')}")
            if chunk["delta"].get("stop_reason") == "refusal":
                print("⚠️  Stream ended with refusal.")

except Exception as e:
    print(f"Error: {e}")

---

## 12. Migrating from Claude Opus 4.8 to Claude Fable 5

Migration is mostly **drop-in**: Claude Fable 5 uses the same Messages API, the same tool use patterns, the same 1M token context window, and the same 128K max output tokens as Claude Opus 4.8. The tokenizer is unchanged, so token counts are roughly the same.

The key differences are:
1. **Always-on adaptive thinking** — thinking cannot be disabled
2. **Thinking output** — raw chain-of-thought is never returned (only summarized)
3. **Safety classifier refusals** — new `stop_reason: "refusal"` response path
4. **Data retention opt-in required** — must set `provider_data_share`
5. **Pricing** — $10/M input, $50/M output (vs $5/$25 for Opus 4.8)

> **Pre-requisite:** If migrating from Claude Opus 4.7 or earlier, first apply the Opus 4.7 → 4.8 migration steps (sampling parameters, manual extended thinking, prefill removal) before this guide.

---

### Step 1: Update model ID

In [ ]:
# Before (Claude Opus 4.8)
# model = "anthropic.claude-opus-4-8"

# After (Claude Fable 5)
# model = "anthropic.claude-fable-5"

# For InvokeModel/Converse (cross-region):
# Before: us.anthropic.claude-opus-4-8
# After:  us.anthropic.claude-fable-5

print("Model ID migration:")
print(f"  Messages API:    anthropic.claude-opus-4-8 → {MODEL_ID}")
print(f"  InvokeModel/Converse: us.anthropic.claude-opus-4-8 → {CRIS_MODEL_ID}")

### Step 2: Handle always-on adaptive thinking

On Claude Opus 4.8, requests without a `thinking` field run without thinking. On Claude Fable 5, **all requests run with adaptive thinking** — it cannot be disabled.

**Impact:**
- `thinking: {type: "disabled"}` → returns a **400 error**
- `thinking: {type: "enabled", budget_tokens: N}` → returns a **400 error**
- Omitting `thinking` entirely → runs with adaptive thinking (unlike Opus 4.8 which ran without thinking)
- `max_tokens` is a hard limit on **total output** (thinking + response text) — revisit it for workloads that ran without thinking on Opus 4.8

Use the `effort` parameter to control thinking depth (`high`, `medium`, `low`).

In [ ]:
# Before (Claude Opus 4.8 — explicit adaptive thinking)
# message = mantle_client.messages.create(
#     model="anthropic.claude-opus-4-8",
#     max_tokens=16000,
#     thinking={"type": "adaptive"},
#     extra_body={"output_config": {"effort": "high"}},
#     messages=[{"role": "user", "content": "..."}],
# )

# After (Claude Fable 5 — thinking is always adaptive, no need to specify)
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MODEL_ID,
            max_tokens=16000,  # Revisit: now includes thinking tokens
            extra_body={"output_config": {"effort": "high"}},
            messages=[{"role": "user", "content": "Explain the observer pattern in 3 sentences."}],
        )

        # Response always contains ThinkingBlock + TextBlock
        for block in message.content:
            if block.type == "thinking":
                print(f"🧠 [Thinking block] — always present on Fable 5")
            elif block.type == "text":
                print(f"💬 {block.text}")

        print(f"\n📊 Usage: Input={message.usage.input_tokens}, Output={message.usage.output_tokens}")
        if hasattr(message.usage, 'output_tokens_details'):
            print(f"   Thinking tokens: {message.usage.output_tokens_details.thinking_tokens}")

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

### Step 3: Handle refusals as a primary code path

Claude Fable 5 runs safety classifiers that can refuse requests. This is **new behavior** if migrating from Opus 4.8.

- `stop_reason: "refusal"` is a successful HTTP 200 response, not an error
- `stop_details.category` reports which classifier fired (`"cyber"`, `"bio"`, `"reasoning_extraction"`, or `null`)
- Prompt-stage refusals (before any output) are **not billed**
- Mid-stream refusals (after partial output) are billed for tokens before the block — discard partial output

**Pattern:**
```python
if message.stop_reason == "refusal":
    # Log, fall back to Opus 4.8, or inform user
    category = getattr(message.stop_details, "category", None)
    ...
elif message.stop_reason == "end_turn":
    # Normal response — extract text blocks
    ...
```

### Step 4: Adapt to thinking output changes

On Claude Fable 5:
- The **raw chain of thought is never returned** — thinking blocks carry only summarized text (when `thinking.display` is set to `"summarized"`)
- Thinking blocks still have a `signature` field — pass them back **unchanged** when continuing a conversation
- `ThinkingBlock` has `.thinking` and `.signature` attributes, **not** `.text` — accessing `.text` on a `ThinkingBlock` raises `AttributeError`

**Always filter content blocks by type:**
```python
# ✅ Correct
for block in message.content:
    if block.type == "text":
        print(block.text)

# ❌ Will break on Fable 5
print(message.content[0].text)  # content[0] is ThinkingBlock
```

### Step 5: Adjust effort level

| Opus 4.8 recommendation | Fable 5 recommendation |
|------------------------|------------------------|
| `xhigh` for coding/high-autonomy | `high` as default for most tasks |
| `high` for general use | `high` for general use |
| N/A | Reserve `xhigh` for most capability-sensitive workloads |

Lower effort settings on Fable 5 still perform well and often exceed `xhigh` performance on prior models. Reduce effort if a task completes but takes longer than necessary.

### Step 6: Note prompt caching differences

| Property | Opus 4.8 | Fable 5 (Bedrock) |
|----------|----------|-------------------|
| Min cacheable tokens | 1,024 | 1,024 |
| Max checkpoints | 4 | 4 |
| TTL options | 5 min, 1 hour | 5 min, 1 hour |



### Migration checklist

| # | Action | Required? |
|---|--------|----------|
| 1 | Update model ID to `anthropic.claude-fable-5` (or CRIS equivalent) | ✅ Yes |
| 2 | Set data retention to `provider_data_share` (one or both endpoints depending on APIs used) | ✅ Yes |
| 3 | Remove `thinking: {type: "disabled"}` or `{type: "enabled", budget_tokens: N}` | ✅ If present |
| 4 | Increase `max_tokens` to account for thinking tokens | ⚠️ Review |
| 5 | Add `stop_reason == "refusal"` handling | ✅ Yes |
| 6 | Replace `message.content[0].text` with block-type iteration | ✅ Yes |
| 7 | Pass thinking blocks back unchanged in multi-turn | ✅ If multi-turn |
| 8 | Adjust effort level (start with `high`) | ⚠️ Review |
| 9 | Budget for ~2x pricing ($10/$50 vs $5/$25 per M tokens) | ⚠️ Review |

---

## 15. Error handling best practices

---

## Summary

Claude Fable 5 is Anthropic's next-generation model for sustained autonomous work — planning, coding, and knowledge synthesis across multi-day tasks.

### Key takeaways

1. **Opt-in required** — Set `provider_data_share` via the Data Retention API before first use
2. **Adaptive thinking only** — Use `thinking.type: "adaptive"` with `output_config.effort`; do NOT use `"enabled"` or `"disabled"`
3. **Handle refusals** — Branch on `stop_reason: "refusal"`; higher refusal rates than previous models
4. **Fallback credit (beta)** — Use `fallback-credit-2026-06-09` beta flag to avoid double-paying cache costs on retries
5. **Connector text summarization** — Thinking blocks between tool calls must be passed back unchanged
6. **Sampling constraints** — `temperature` must be 1.0, `top_p` ≥ 0.99 and < 1.0, `top_k` not supported

### Resources

- [Model Card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-fable-5.html)
- [Anthropic Documentation](https://www.anthropic.com/news/claude-fable-5-mythos-5)
- [Claude Adaptive Thinking](https://docs.aws.amazon.com/bedrock/latest/userguide/claude-messages-adaptive-thinking.html)
- [Data Retention API](https://docs.aws.amazon.com/bedrock/latest/userguide/abuse-detection.html)